# Road Damage RAG — branchement du tableau de bord TariqMap

Ce notebook construit l'index documentaire et expose `POST /ask`.

Le tableau de bord ne lance pas Colab. Après la dernière cellule :

1. Copiez uniquement la ligne `VITE_RAG_API_URL=https://….ngrok-free.app`. N'ajoutez pas `/ask`.
2. Collez-la dans `dashboard/.env.local`.
3. Arrêtez `npm run dev`, puis relancez-le. Vite lit cette variable seulement au démarrage.
4. Dans Colab, créez le secret `NGROK_AUTH_TOKEN` avec votre jeton ngrok (ngrok.com → Your Authtoken). Ce n'est pas une clé Supabase.

Une adresse affichée lors d'une session précédente est périmée. Runtime : GPU T4. Les fichiers `.txt` du corpus vont dans `/content/road_damage_rag/data` avant le chargement des documents.


In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA :", torch.version.cuda)
else:
    print("GPU non disponible")

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu

In [ ]:
import torch

print("CUDA :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu"
)

print("BGE-M3 chargé avec succès !")

In [ ]:
texts = [
    "Les fissures longitudinales sont orientées parallèlement à l'axe de la chaussée.",
    "Les nids-de-poule sont des dégradations localisées de la chaussée.",
    "La maintenance routière comprend différentes méthodes de réparation."
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print("Shape :", embeddings.shape)

In [ ]:
query = "Qu'est-ce qu'une fissure longitudinale ?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

similarities = embeddings @ query_embedding.T

print(similarities)

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

print("Qwen chargé avec succès !")

In [ ]:
print(
    f"GPU utilisé : {torch.cuda.get_device_name(0)}"
)

print(
    f"VRAM utilisée : "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Tu es un assistant spécialisé dans les dommages routiers."
    },
    {
        "role": "user",
        "content": "Explique brièvement ce qu'est une fissure longitudinale dans une route."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    [text],
    return_tensors="pt"
).to(llm.device)

with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

In [ ]:
documents = [
    """
    Les fissures longitudinales sont des fissures orientées
    parallèlement à l'axe de la chaussée. Elles peuvent être
    associées à des problèmes de structure ou de vieillissement.
    """,

    """
    Les nids-de-poule sont des cavités localisées dans la chaussée.
    Leur formation peut être favorisée par l'eau, le trafic et
    la dégradation progressive du revêtement.
    """,

    """
    La maintenance routière comprend l'inspection, la détection
    des dégradations, la réparation et le suivi de l'état
    des infrastructures.
    """
]

In [ ]:
doc_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

In [ ]:
question = "Quels sont les problèmes associés aux fissures longitudinales ?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

In [ ]:
scores = doc_embeddings @ question_embedding.T

best_index = scores[:, 0].argmax()

print("Document sélectionné :")
print(documents[best_index])

print("Score :", scores[best_index, 0])

Etape 6 : FAISS

In [ ]:
import faiss
import numpy as np

# FAISS utilise des float32
doc_embeddings = np.asarray(
    doc_embeddings,
    dtype="float32"
)

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings)

print("Nombre de vecteurs :", index.ntotal)
print("Dimension :", dimension)

In [ ]:
question = "Quels sont les problèmes associés aux fissures longitudinales ?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

question_embedding = np.asarray(
    question_embedding,
    dtype="float32"
)

scores, indices = index.search(
    question_embedding,
    k=2
)

print("Indices :", indices)
print("Scores :", scores)

In [ ]:
for rank, idx in enumerate(indices[0]):
    print(f"\n--- Résultat {rank + 1} ---")
    print(documents[idx])
    print("Score :", scores[0][rank])

In [ ]:
import os

BASE_DIR = "/content/road_damage_rag"
DATA_DIR = os.path.join(BASE_DIR, "data")
VECTOR_DIR = os.path.join(BASE_DIR, "vector_db")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VECTOR_DIR, exist_ok=True)

print("Dossiers créés.")

In [ ]:
import os

for filename in os.listdir(DATA_DIR):
    print(filename)

## Manuels à uploader

`/content/road_damage_rag/data` est créé vide. Sans les six `.txt`, l'index FAISS plante avec `IndexError: tuple index out of range` sur `chunk_embeddings.shape[1]`.

La cellule suivante ouvre le sélecteur de fichiers Colab. Choisissez les six manuels, puis continuez.


In [ ]:
import os
import shutil

from google.colab import files

DATA_DIR = "/content/road_damage_rag/data"
os.makedirs(DATA_DIR, exist_ok=True)

EXPECTED = [
    "Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt",
    "Pathologie_des_routes_contenu_organise.txt",
    "Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt",
    "Distress_Identification_Manual_contenu_organise.txt",
    "RDD2022_contenu_organise.txt",
    "ArticleOuargla_contenu_organise.txt",
]

# Recover files already dropped somewhere under /content.
for dirpath, _dirnames, filenames in os.walk("/content"):
    if os.path.abspath(dirpath) == os.path.abspath(DATA_DIR):
        continue
    for filename in filenames:
        if filename not in EXPECTED:
            continue
        source = os.path.join(dirpath, filename)
        target = os.path.join(DATA_DIR, filename)
        if not os.path.exists(target):
            shutil.copy2(source, target)
            print("Copié :", source)

present = [
    name for name in sorted(os.listdir(DATA_DIR))
    if name.lower().endswith(".txt")
]
missing = [name for name in EXPECTED if name not in present]

if missing:
    print("Il manque", len(missing), "manuel(s). Sélectionnez les fichiers .txt.")
    uploaded = files.upload()
    for name, content in uploaded.items():
        if not name.lower().endswith(".txt"):
            print("Ignoré, ce n'est pas un .txt :", name)
            continue
        target = os.path.join(DATA_DIR, os.path.basename(name))
        with open(target, "wb") as handle:
            handle.write(content)
        print("Enregistré :", target, len(content), "octets")

present = [
    name for name in sorted(os.listdir(DATA_DIR))
    if name.lower().endswith(".txt")
]
if not present:
    raise RuntimeError(
        "Aucun .txt dans /content/road_damage_rag/data. "
        "Uploadez les six manuels, puis relancez cette cellule. "
        "Ne continuez pas : FAISS n'a aucun vecteur à indexer."
    )

print("Manuels prêts :", len(present))
for name in present:
    size = os.path.getsize(os.path.join(DATA_DIR, name))
    print(f"- {name} ({size} octets)")


In [ ]:
import os

DATA_DIR = "/content/road_damage_rag/data"

documents = []

for filename in sorted(os.listdir(DATA_DIR)):
    if not filename.lower().endswith(".txt"):
        continue
    path = os.path.join(DATA_DIR, filename)
    with open(path, "r", encoding="utf-8") as handle:
        text = handle.read()
    if not text.strip():
        print("Ignoré, fichier vide :", filename)
        continue
    documents.append({
        "text": text,
        "source": filename,
    })

print("Documents chargés :", len(documents))
for doc in documents:
    print("-", doc["source"])

if not documents:
    raise RuntimeError(
        "Aucun manuel chargé depuis /content/road_damage_rag/data. "
        "Relancez la cellule d'upload des .txt, puis relancez cette cellule. "
        "Sans document, chunk_embeddings n'a pas de dimension et FAISS lève "
        "IndexError: tuple index out of range."
    )


Découper les documents en chunks

Maintenant on découpe chaque document en morceaux.

Je te conseille de commencer avec :

chunk_size = 800
overlap = 100

In [ ]:
def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = []

for doc in documents:

    doc_chunks = chunk_text(
        doc["text"],
        chunk_size=800,
        overlap=100
    )

    for chunk in doc_chunks:

        chunks.append({
            "text": chunk,
            "source": doc["source"]
        })

print("Nombre total de chunks :", len(chunks))

In [ ]:
for i in range(min(3, len(chunks))):

    print("=" * 60)
    print("CHUNK", i + 1)
    print("SOURCE :", chunks[i]["source"])
    print(chunks[i]["text"][:500])

Embeddings avec BGE-M3

Maintenant on transforme les chunks, et non les documents entiers, en vecteurs.

In [ ]:
texts = [chunk["text"] for chunk in chunks]

if not texts:
    raise RuntimeError(
        "Aucun chunk à encoder. Les manuels .txt n'ont pas été chargés "
        "depuis /content/road_damage_rag/data."
    )

chunk_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")
if chunk_embeddings.ndim == 1:
    chunk_embeddings = chunk_embeddings.reshape(1, -1)

print("Shape :", chunk_embeddings.shape)
if chunk_embeddings.ndim != 2 or chunk_embeddings.shape[0] == 0:
    raise RuntimeError(
        "Les embeddings n'ont pas la forme (nombre de chunks, dimension) : "
        f"{chunk_embeddings.shape}."
    )


In [ ]:
import numpy as np

chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32"
)

print("Shape :", chunk_embeddings.shape)

Construire FAISS

Maintenant on donne ces vecteurs à FAISS :

In [ ]:
import faiss
import numpy as np

chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")
if getattr(chunk_embeddings, "ndim", 0) != 2 or chunk_embeddings.shape[0] == 0:
    raise RuntimeError(
        "Aucun embedding à indexer, shape="
        f"{getattr(chunk_embeddings, 'shape', None)}. "
        "Le dossier /content/road_damage_rag/data ne contenait pas les manuels "
        "au moment du calcul. Uploadez les six .txt, puis relancez depuis la "
        "cellule d'upload. Ne relancez pas seulement cette cellule."
    )

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("FAISS prêt.")
print("Nombre de vecteurs :", index.ntotal)
print("Dimension :", dimension)


In [ ]:
def retrieve(query, k=5):
    """Retourne au plus k passages. k est borné à 1..8, comme le tableau de bord."""
    if index.ntotal == 0:
        return []

    try:
        k = int(k)
    except (TypeError, ValueError):
        k = 5
    k = min(8, max(1, k), int(index.ntotal))

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )
    query_embedding = np.asarray(query_embedding, dtype="float32")

    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        idx = int(idx)
        if idx < 0 or idx >= len(chunks):
            continue
        text = chunks[idx].get("text")
        source = str(chunks[idx].get("source") or "").strip()
        if not source or not isinstance(text, str):
            continue
        results.append({
            "text": text,
            "source": source,
            "score": float(score),
        })

    return results


In [ ]:
question = "Qu'est-ce qu'une fissure longitudinale ?"

results = retrieve(question, k=5)

for i, result in enumerate(results):

    print("=" * 70)
    print(f"RESULTAT {i+1}")
    print("Source :", result["source"])
    print("Score :", round(result["score"], 4))
    print()
    print(result["text"][:800])

In [ ]:
def build_context(results):

    context = ""

    for i, result in enumerate(results):

        context += f"""
SOURCE {i+1}: {result['source']}

{result['text']}

------------------------------
"""

    return context

In [ ]:
context = build_context(results)

print(context)

In [ ]:
def build_prompt(question, context):

    return f"""
Tu es un assistant spécialisé dans les dommages et pathologies routières.

Réponds à la question uniquement à partir du contexte fourni.

Règles :
- Utilise uniquement les informations présentes dans le contexte.
- N'invente aucune information.
- Si l'information n'est pas présente dans le contexte,
  indique clairement que l'information n'a pas été trouvée.
- Donne une réponse claire, précise et concise.
- Ne mentionne pas les sources dans ta réponse.
- Ne mentionne pas "SOURCE 1", "SOURCE 2", etc.
- Les sources seront affichées séparément par l'application.

Contexte :
{context}

Question :
{question}

Réponse :
"""

In [ ]:
def generate_answer(question, context):

    prompt = build_prompt(
        question,
        context
    )

    messages = [
        {
            "role": "system",
            "content": "Tu es un assistant spécialisé dans les dommages routiers."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(llm.device)

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [ ]:
def ask_rag(question, k=5):

    # 1. Recherche des chunks
    results = retrieve(
        question,
        k=k
    )

    # 2. Construction du contexte
    context = build_context(
        results
    )

    # 3. Génération avec Qwen
    answer = generate_answer(
        question,
        context
    )

    return answer, results

Premier vrai test

In [ ]:
question = "Qu'est-ce qu'une fissure longitudinale et quelles sont ses caractéristiques ?"

answer, sources = ask_rag(
    question,
    k=5
)

print("========== RÉPONSE ==========")
print(answer)

print("\n========== SOURCES ==========")

for source in sources:

    print(
        source["source"],
        "→ score :",
        round(source["score"], 4)
    )

In [ ]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)

In [ ]:
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

In [ ]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)
print("First parameter :", next(llm.parameters()).device)

In [ ]:
# import gc
# import torch

# del llm
# gc.collect()
# torch.cuda.empty_cache()

# print("VRAM après nettoyage :", torch.cuda.memory_allocated() / 1024**3, "GB")

In [ ]:
# Ne pas recharger Qwen ici.
# Un second from_pretrained sur le T4 sature la VRAM et empêche d'atteindre l'API.
# Le modèle chargé plus haut est celui utilisé par ask_rag.
print("Modèle conservé :", model_name)
print("Device :", llm.device)


In [ ]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)
print("First parameter :", next(llm.parameters()).device)

# l’étape des tests RAG.

In [ ]:
test_questions = [
    "Qu'est-ce qu'une fissure longitudinale et quelles sont ses caractéristiques ?",

    "Qu'est-ce qu'une fissure transversale ?",

    "Quelles sont les principales causes des fissures de chaussée ?",

    "Quelle est la différence entre une fissure longitudinale et une fissure transversale ?",

    "Qu'est-ce que le faïençage d'une chaussée ?",

    "Qu'est-ce qu'une ornière et comment se manifeste-t-elle ?",

    "Quels sont les principaux types de dégradations des chaussées ?",

    "Comment identifier une dégradation de la chaussée à partir de son aspect visuel ?",

    "Quelles sont les classes de dommages présentes dans RDD2022 ?",

    "Quelle est la différence entre longitudinal crack et lateral crack ?"
]

for i, question in enumerate(test_questions, 1):

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print("QUESTION :", question)

    answer, sources = ask_rag(
        question,
        k=5
    )

    print("\nRÉPONSE :")
    print(answer)

    print("\nSOURCES RÉCUPÉRÉES :")

    for j, source in enumerate(sources, 1):
        print(
            f"{j}. {source['source']} "
            f"→ score : {source['score']:.4f}"
        )

on passe au sauvegarde du RAG

sauvegarde le faisse

In [ ]:
import os
import faiss

VECTOR_DIR = "/content/road_damage_rag/vector_db"

os.makedirs(VECTOR_DIR, exist_ok=True)

faiss.write_index(
    index,
    os.path.join(VECTOR_DIR, "index.faiss")
)

print("Index FAISS sauvegardé.")

sauvegarde le chunk

In [ ]:
import pickle
import os

chunks_path = os.path.join(
    VECTOR_DIR,
    "chunks.pkl"
)

with open(chunks_path, "wb") as f:
    pickle.dump(chunks, f)

print("Chunks sauvegardés.")

Sauvegarder la configuration

C'est utile pour savoir comment l'index a été construit.

In [ ]:
import json

config = {
    "embedding_model": "BAAI/bge-m3",
    "llm_model": "Qwen/Qwen2.5-7B-Instruct",
    "chunk_size": 800,
    "chunk_overlap": 100,
    "embedding_dimension": int(chunk_embeddings.shape[1]),
    "num_chunks": len(chunks),
    "retrieval_k": 5
}

config_path = os.path.join(
    VECTOR_DIR,
    "config.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Configuration sauvegardée.")

Vérifier les fichiers

In [ ]:
import os

print("Contenu de vector_db :")

for file in os.listdir(VECTOR_DIR):
    path = os.path.join(VECTOR_DIR, file)
    print(
        file,
        "→",
        round(os.path.getsize(path) / 1024 / 1024, 2),
        "MB"
    )

## API pour le tableau de bord

`POST /ask` reçoit `{"question": "...", "k": 5}` et répond `{"answer": "...", "sources": [{"source", "score", "text"}]}`.

Le tableau de bord appelle `/rag/ask` sur son propre serveur. Vite retire `/rag` et transmet la requête à cette origine. Le navigateur ne contacte pas ngrok directement.


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field


app = FastAPI(
    title="Road Damage RAG API",
    description="API RAG pour les dommages routiers, appelée par le tableau de bord TariqMap",
    version="1.1",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)
    k: int = Field(default=5, ge=1, le=8)


@app.get("/")
def root():
    return {
        "status": "ok",
        "message": "Road Damage RAG API is running",
    }


@app.post("/ask")
def ask(request: QuestionRequest):
    answer, sources = ask_rag(request.question.strip(), k=request.k)
    answer = (answer or "").strip()
    if not answer:
        answer = "L'information n'a pas été trouvée dans les manuels."

    cleaned = []
    for source in sources:
        name = str(source.get("source") or "").strip()
        text = source.get("text")
        score = source.get("score")
        if not name or not isinstance(text, str) or not isinstance(score, (int, float)):
            continue
        cleaned.append({
            "source": name,
            "score": float(score),
            "text": text,
        })

    return {
        "answer": answer,
        "sources": cleaned,
    }


In [ ]:
import socket
import threading
import time
import urllib.request

import uvicorn


def port_open(port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.4)
        return sock.connect_ex(("127.0.0.1", port)) == 0


def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


if port_open():
    print("L'API répond déjà sur le port 8000.")
else:
    threading.Thread(target=run_api, daemon=True).start()
    for _ in range(40):
        if port_open():
            break
        time.sleep(0.25)
    else:
        raise RuntimeError("L'API n'a pas démarré sur le port 8000.")

with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=5) as response:
    print(response.read().decode())


In [ ]:
import os

from pyngrok import ngrok


def read_ngrok_token():
    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        return str(userdata.get("NGROK_AUTH_TOKEN") or "").strip()
    except Exception:
        return ""


token = read_ngrok_token()
if not token:
    raise RuntimeError(
        "Secret Colab manquant : NGROK_AUTH_TOKEN. "
        "Ajoutez le jeton ngrok (Your Authtoken), puis relancez cette cellule. "
        "Ne collez pas le jeton dans le notebook."
    )

ngrok.set_auth_token(token)

for tunnel in list(ngrok.get_tunnels()):
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000, bind_tls=True)
origin = getattr(public_url, "public_url", str(public_url)).strip().rstrip("/")

print("API publique :", origin)
print()
print("Collez cette ligne dans dashboard/.env.local, puis relancez npm run dev :")
print(f"VITE_RAG_API_URL={origin}")
